# Submission Klasifikasi Dokumen

## Import repository dan konfigurasi

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
repo_candidates = list(INPUT_ROOT.rglob("scripts/config.py"))
if not repo_candidates:
    raise FileNotFoundError(
        "Repository belum ditambahkan sebagai Kaggle Input. "
        "Pilih Add Input dan tambahkan dataset repository."
    )
REPO_ROOT = repo_candidates[0].parent.parent

csv_candidates = list(INPUT_ROOT.rglob("train.csv"))
csv_candidates = [p for p in csv_candidates if (p.parent / "test.csv").exists()
                  and (p.parent / "sample_submission.csv").exists()]
if not csv_candidates:
    raise FileNotFoundError("Folder kompetisi berisi train.csv/test.csv/sample_submission.csv tidak ditemukan.")
DATA_ROOT = csv_candidates[0].parent

weight_candidates = list(INPUT_ROOT.rglob("efficientnet_b0_rwightman-7f5810bc.pth"))
os.environ["DOC_DATA_DIR"] = str(DATA_ROOT)
os.environ["DOC_CHECKPOINT_DIR"] = "/kaggle/working/checkpoint"
os.environ["DOC_SUBMISSION_DIR"] = "/kaggle/working/submissions"
os.environ["DOC_REPORT_DIR"] = "/kaggle/working/reports"
if weight_candidates:
    os.environ["DOC_WEIGHTS_PATH"] = str(weight_candidates[0])

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

print("Repository:", REPO_ROOT)
print("Dataset:", DATA_ROOT)
print("Pretrained weights:", weight_candidates[0])
print("Output: /kaggle/working")

from scripts.config import EXPERIMENTS


## Prediksi baseline, tuning 1, dan tuning 2

In [ ]:

# Pastikan checkpoint tiga eksperimen sudah ada di /kaggle/working/checkpoint.
subprocess.run(
    [sys.executable, "-m", "scripts.predict", "--experiment", "all",
     "--data-dir", str(DATA_ROOT), "--batch-size", "16", "--num-workers", "2"],
    check=True
)


## Validasi submission

In [ ]:

submission_dir = Path("/kaggle/working/submissions")
for filename in ["submission_baseline.csv", "submission_tuning.csv", "submission_tuning2.csv"]:
    path = submission_dir / filename
    df = pd.read_csv(path)
    assert df.shape == (7994, 2)
    assert list(df.columns) == ["id", "label"]
    assert df["label"].between(0, 15).all()
    print(filename, df.shape, "VALID")
